# Lezione 7: Modelli di Controllo e Studio di Ablazione Scientifica
**File**: [`models/baselines.py`](file:///g:/My%20Drive/PhD%20&%20Research/Temporal%20GNN/cfm_grn/models/baselines.py)  
**Ruolo nel progetto**: Fornisce le tre baseline canoniche necessarie per dimostrare la validità scientifica dell'approccio.


---
## 1. Il Rigore Metodologico nella Ricerca di Dottorato

Per pubblicare su riviste ad alto impatto (es. *Nature Methods* o conferenze come *NeurIPS/ICML*), non basta mostrare che un modello funziona. Bisogna dimostrare **PERCHÉ** funziona tramite uno studio di ablazione rigoroso su tre ipotesi:

| Modello | Domanda Scientifica Risolta | Cosa rimuove? |
|---|---|---|
| **GraphVectorField** (Nostro) | Modello completo con prior e maschera dinamica | - |
| **StaticGraphVectorField** | Il prior biologico statico basta, o serve modulare i pesi $M_\theta$? | Rimuove la maschera dinamica (pesi fissi a 1.0) |
| **MLPVectorField** | Il grafo biologico serve davvero, o un MLP denso può impararlo da solo? | Rimuove completamente il grafo di regolazione |
| **PersistenceBaseline** | Il modello sta davvero predicendo la transizione o sta solo copiando lo stato iniziale? | Velocità nulla ($x_1 = x_0$) |


---
## 2. Setup e Import


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
from models.graph_vector_field import GraphVectorField
from models.baselines import MLPVectorField, StaticGraphVectorField, PersistenceBaseline
from data.synthetic_grn import SyntheticGRNSimulator
print('Tutte le baseline caricate!')


---
## 3. Confronto del Numero di Parametri

Grazie al vincolo sparso della rete di regolazione genica, la GNN può operare con parametri mirati senza rischiare l'overfitting dei modelli densi globali.


In [ ]:
sim = SyntheticGRNSimulator(num_genes=20, density=0.25, seed=42)

m_graph = GraphVectorField(num_genes=20, edge_index=sim.edge_index, perturbation_dim=20, hidden_dim=48)
m_static = StaticGraphVectorField(num_genes=20, edge_index=sim.edge_index, perturbation_dim=20, hidden_dim=48)
m_mlp = MLPVectorField(num_genes=20, perturbation_dim=20, hidden_dim=128, num_layers=3)

def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

print(f'Parametri Graph_CFM (Nostro):        {count_params(m_graph):,}')
print(f'Parametri Static_Graph_CFM:         {count_params(m_static):,}')
print(f'Parametri MLP_CFM (Senza Grafo):    {count_params(m_mlp):,}')


---
## 4. Test di Compatibilità Interfaccia

Tutti i modelli implementano la medesima firma `forward(x, t, perturbation)` per consentire l'addestramento trasparente nel ciclo di training.


In [ ]:
B = 4
x_test = torch.randn(B, 20)
t_test = torch.rand(B, 1)
p_test = torch.zeros(B, 20)

out_g = m_graph(x_test, t_test, p_test)
out_s = m_static(x_test, t_test, p_test)
out_m = m_mlp(x_test, t_test, p_test)

print('Shape Graph CFM: ', out_g.shape)
print('Shape Static CFM:', out_s.shape)
print('Shape MLP CFM:   ', out_m.shape)
assert out_g.shape == out_s.shape == out_m.shape == (B, 20), 'Incompatibilità di shape!'
